In [28]:
import lightgbm

from config import tick_size
from data_loader import CONFIG_FILES_DIR
from src.data_loader import csv_to_parquet, RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR
from src.config import start_date
from features import MarketDemandModel_data_transformation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from category_encoders import TargetEncoder

import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error

In [29]:
df = pd.read_parquet(PROCESSED_DATA_DIR / 'online_retail.parquet')

In [30]:
X_train , y_train = MarketDemandModel_data_transformation(df)

In [21]:
print(X_train)

            StockCode  Quantity  Revenue  Unique_Customers      Price  lag_1t  \
InvoiceDate                                                                     
2009-12-01      15036        55    35.75                 2   0.650000     0.0   
2009-12-01      21122        32    41.32                 5   1.291250     0.0   
2009-12-01      82583         3     8.50                 1   2.833333     0.0   
2009-12-01      21121        48    60.00                 2   1.250000     0.0   
2009-12-01      82600        37    77.70                 3   2.100000     0.0   
...               ...       ...      ...               ...        ...     ...   
2011-12-08      22029        32    16.72                 2   0.522500   228.0   
2011-12-08      22411        60   171.95                 4   2.865833    65.0   
2011-12-08      22197      3111  2507.55                 5   0.806027   368.0   
2011-12-08      22027        36    15.12                 3   0.420000    12.0   
2011-12-08       POST       

In [31]:
tscv = TimeSeriesSplit(n_splits=5)

X = X_train.sort_index(level='InvoiceDate')
y = y_train.sort_index(level='InvoiceDate').clip(upper = y_train.quantile(0.99))

params = {
    'objective': 'regression_l1',
    'metric': 'l1',
    'force_row_wise': True,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbosity': -1
}

cv_mae_scores = []

for fold, (train_index, val_index) in enumerate(tscv.split(X)):
    print(f"--- Training Fold {fold+1} ---")


    X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
    y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

    encoder = TargetEncoder(cols=['StockCode'], smoothing=10)
    X_train_fold = encoder.fit_transform(X_train_fold, y_train_fold)
    X_val_fold = encoder.transform(X_val_fold)

    X_train_fold.rename(columns={'StockCode': 'StockCode_enc'}, inplace=True)
    X_val_fold.rename(columns={'StockCode': 'StockCode_enc'}, inplace=True)

    train_data = lgb.Dataset(X_train_fold, label=y_train_fold)
    valid_data = lgb.Dataset(X_val_fold, label=y_val_fold, reference=train_data)

    MarketDemandModel = lgb.train(
        params,
        train_data,
        valid_sets=[train_data, valid_data],
        valid_names=['train', 'valid'],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(stopping_rounds=50)]
    )

    best_fold_mae = MarketDemandModel.best_score['valid']['l1']
    cv_mae_scores.append(best_fold_mae)

--- Training Fold 1 ---
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[137]	train's l1: 17.0451	valid's l1: 18.9946
--- Training Fold 2 ---
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[202]	train's l1: 17.7384	valid's l1: 17.0365
--- Training Fold 3 ---
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[137]	train's l1: 17.3583	valid's l1: 15.9541
--- Training Fold 4 ---
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[95]	train's l1: 16.9223	valid's l1: 15.434
--- Training Fold 5 ---
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[119]	train's l1: 16.5625	valid's l1: 17.4198


In [32]:
print(f"Target Mean: {y.mean():.2f}")
print(f"Target Median: {y.median():.2f}")
print(f"Target Max: {y.max():.2f}")
print(f"Target Std Dev: {y.std():.2f}")

Target Mean: 21.35
Target Median: 6.00
Target Max: 268.00
Target Std Dev: 42.49


In [33]:
print("="*40)
print(f"Mean Validation MAE across {tscv.n_splits} folds: {np.mean(cv_mae_scores):.4f}")
print(f"Standard Deviation:  {np.std(cv_mae_scores):.4f}")
print("="*40)

Mean Validation MAE across 5 folds: 16.9678
Standard Deviation:  1.2407


In [36]:
naive_errors = np.abs(y - X['lag_1t'])

print("========================================")
print(f"Global Naive Baseline MAE: {naive_errors.mean():.4f}")
print("========================================")

Global Naive Baseline MAE: 30.2272


In [37]:
print("Training Final Production Model")

master_encoder = TargetEncoder(cols=['StockCode'], smoothing=10)
X_final = master_encoder.fit_transform(X, y)
X_final.rename(columns={'StockCode': 'StockCode_enc'}, inplace=True)

joblib.dump(master_encoder, CONFIG_FILES_DIR / 'MarketDemandModelEncoder.joblib')

final_train_data = lgb.Dataset(X_final, label=y)

final_model = lgb.train(
    params,
    final_train_data,
    num_boost_round=150
)



Training Final Production Model


In [38]:
joblib.dump(final_model, MODELS_DIR / 'MarketDemandModel.joblib')

['C:\\Users\\UserB\\Documents\\ML\\Online Retail II UCI\\models\\MarketDemandModel.joblib']

In [ ]:
df['StockCode']